In [ ]:
#Visualizations
# Install plotting libraries
!pip -q install matplotlib numpy pandas scikit-learn

import numpy as np
import matplotlib.pyplot as plt

#Learning curve (Logistic Regression) using accuracy on validation set
fractions = [0.1, 0.2, 0.3, 0.5, 0.7, 1.0]
lc_scores = []

# Use a simple LR pipeline without CV for learning curve to keep it fast
lr_lc = LogisticRegression(featuresCol="features", labelCol="labelIndex", maxIter=30)
pipe_lr_lc = Pipeline(stages=[label_indexer, assembler, lr_lc])

for frac in fractions:
    tdf = train_df.sample(False, frac, 42)
    model_lc = pipe_lr_lc.fit(tdf)
    pred_lc = model_lc.transform(valid_df)
    lc_scores.append(float(acc_eval.evaluate(pred_lc)))

plt.figure()
plt.plot(fractions, lc_scores, marker="o")
plt.xlabel("Training data fraction")
plt.ylabel("Validation Accuracy")
plt.title("Learning Curve (Logistic Regression)")
plt.grid(True)
plt.show()

#Confusion matrix (Logistic Regression predictions)
from pyspark.mllib.evaluation import MulticlassMetrics

cm_rdd = lr_test.select("prediction", "labelIndex").rdd.map(lambda r: (float(r[0]), float(r[1])))
metrics = MulticlassMetrics(cm_rdd)
cm = metrics.confusionMatrix().toArray()

plt.figure(figsize=(6, 5))
plt.imshow(cm, interpolation="nearest")
plt.title("Confusion Matrix (Logistic Regression)")
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.colorbar()
plt.show()

#ROC curve and Precision-Recall curve (binary: most frequent label vs others)
# Determine most frequent label index in test
top_row = lr_test.groupBy("labelIndex").count().orderBy(F.desc("count")).limit(1).collect()[0]
pos_label_index = int(top_row["labelIndex"])

# Collect a sample for plotting curves
sample_n = 20000
plot_df = lr_test.select("labelIndex", "probability").orderBy(F.rand(seed=7)).limit(sample_n).collect()

y_true = []
y_score = []

for r in plot_df:
    y_true.append(1 if int(r["labelIndex"]) == pos_label_index else 0)
    prob_vec = r["probability"]
    y_score.append(float(prob_vec[pos_label_index]))

from sklearn.metrics import roc_curve, auc, precision_recall_curve, average_precision_score

fpr, tpr, _ = roc_curve(y_true, y_score)
roc_auc = auc(fpr, tpr)

plt.figure()
plt.plot(fpr, tpr)
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve (One-vs-Rest, Logistic Regression)  AUC=" + str(round(roc_auc, 4)))
plt.grid(True)
plt.show()

prec, rec, _ = precision_recall_curve(y_true, y_score)
ap = average_precision_score(y_true, y_score)

plt.figure()
plt.plot(rec, prec)
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision-Recall Curve (One-vs-Rest, Logistic Regression)  AP=" + str(round(ap, 4)))
plt.grid(True)
plt.show()

#Feature importance plots (Decision Tree + Random Forest)
# Extract feature importances from fitted pipelines
dt_stage = dt_model.stages[-1]
rf_stage = rf_model.stages[-1]

dt_imp = np.array(dt_stage.featureImportances.toArray())
rf_imp = np.array(rf_stage.featureImportances.toArray())

#Decision Tree feature importance
plt.figure(figsize=(8, 4))
plt.bar(range(len(feature_cols)), dt_imp)
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
plt.title("Feature Importance (Decision Tree)")
plt.tight_layout()
plt.show()

#Random Forest feature importance
plt.figure(figsize=(8, 4))
plt.bar(range(len(feature_cols)), rf_imp)
plt.xticks(range(len(feature_cols)), feature_cols, rotation=45, ha="right")
plt.title("Feature Importance (Random Forest)")
plt.tight_layout()
plt.show()

dfp.unpersist()